In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
# You can change these values
TICKER = 'MSFT'  # Microsoft Corporation
START_DATE = '2015-01-01'
END_DATE = '2023-12-31' # Or pd.Timestamp.now() for current date

print(f"--- Fetching daily data for {TICKER} from {START_DATE} to {END_DATE} ---")

# --- 1. Get Daily Data from yfinance ---
try:
    # Download daily historical data
    # We need 'Adj Close' for accurate returns and 'Volume'
    df_daily = yf.download(TICKER, start=START_DATE, end=END_DATE)
    df_daily = df_daily.columns.droplevel(0, axis=1, inplace=True) # Drop the first level of column names

    if df_daily.empty:
        raise ValueError(f"No data downloaded for {TICKER}. Check ticker symbol and date range.")

    print("\n--- Raw Daily Data Head ---")
    print(df_daily.head())
    print("\n--- Raw Daily Data Info ---")
    df_daily.info()

except Exception as e:
    print(f"Error downloading data: {e}")
    print("Please ensure the ticker symbol is correct and you have an active internet connection.")
    exit() # Exit if data download fails

# --- 2. Prepare Daily Data for Regression ---

# Calculate Daily Return based on 'Adj Close'
# (Today's Adj Close / Yesterday's Adj Close) - 1
df_daily['Daily_Return'] = df_daily['Close'].pct_change()

# Create a column for next day's return
# .shift(-1) moves the values up by 1 position (so current row gets next day's return)
df_daily['Next_Day_Return'] = df_daily['Daily_Return'].shift(-1)

# Drop rows with NaN values. This will remove:
# - The first row (Daily_Return will be NaN)
# - The last row (Next_Day_Return will be NaN)
df_regression = df_daily.dropna()

# Select only the relevant columns for regression
df_regression = df_regression[['Volume', 'Next_Day_Return']]

# print("\n--- Daily Data for Regression Head (aligned) ---")
# print(df_regression.head())
# print("\n--- Daily Data for Regression Tail (aligned) ---")
# print(df_regression.tail())
# print(f"\nOriginal daily rows: {len(df_daily)}, Rows for regression: {len(df_regression)}")

# --- 3. Linear Regression Analysis (using statsmodels) ---

# Define independent variable (X) and dependent variable (Y)
X = df_regression['Volume']
Y = df_regression['Next_Day_Return']

# Add a constant to the independent variable (for the intercept term)
X = sm.add_constant(X)

# Create and fit the OLS (Ordinary Least Squares) model
model = sm.OLS(Y, X)
results = model.fit()

# Print the regression summary
print("\n--- Linear Regression Results (statsmodels) ---")
print(results.summary())
print (results)
# Extract key metrics
r_squared = results.rsquared
p_value_volume = results.pvalues['Volume']
coefficient_volume = results.params['Volume']
intercept = results.params['const']

print(f"\n--- Key Regression Metrics ---")
print(f"R-squared: {r_squared:.6f}") # Display more precision for R-squared, often very low
print(f"P-value for Volume: {p_value_volume:.6f}")
print(f"Coefficient for Volume: {coefficient_volume:.12f}") # Display more precision for small coefficients
print(f"Intercept: {intercept:.6f}")

# Interpretation:
print("\n--- Interpretation ---")
if p_value_volume < 0.05:
    print(f"The relationship between Today's Volume and Tomorrow's Return is statistically significant (p < 0.05).")
    print(f"For every 1 unit increase in Today's Volume, Tomorrow's Return is expected to change by {coefficient_volume:.12f}.")
    if coefficient_volume > 0:
        print("This suggests a positive relationship: higher volume tends to be followed by higher next day's return.")
    else:
        print("This suggests a negative relationship: higher volume tends to be followed by lower next day's return.")
else:
    print(f"The relationship between Today's Volume and Tomorrow's Return is NOT statistically significant (p >= 0.05).")
    print(f"The R-squared value of {r_squared:.6f} indicates that Today's Volume explains a very small portion of the variance in Tomorrow's Return.")
    print("This means daily volume is likely not a good linear predictor of next day's return based on this data.")

print("\n--- Important Note ---")
print("For daily financial data, especially for well-known stocks, finding statistically significant and practically useful")
print("linear relationships with simple predictors like volume is very rare due to market efficiency.")
print("R-squared values will almost always be extremely low, indicating that volume explains very little of the daily return variance.")


# --- 4. Visualization ---

plt.figure(figsize=(12, 7))
sns.scatterplot(x='Volume', y='Next_Day_Return', data=df_regression, alpha=0.5, s=10) # s=10 to make points smaller

# Plot the regression line
x_vals = np.array([df_regression['Volume'].min(), df_regression['Volume'].max()])
y_vals = intercept + coefficient_volume * x_vals
plt.plot(x_vals, y_vals, color='red', linestyle='--', label='Regression Line')


plt.title(f'Today\'s Volume vs. Tomorrow\'s Return for {TICKER}')
plt.xlabel('Today\'s Volume')
plt.ylabel('Tomorrow\'s Return')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Optional: Residuals plot to check model assumptions
residuals = results.resid

plt.figure(figsize=(12, 6))
sns.residplot(x=df_regression['Volume'], y=df_regression['Next_Day_Return'], lowess=True, line_kws={'color': 'red', 'lw': 1, 'alpha': 0.8})
plt.title(f'Residuals Plot for {TICKER} Daily Regression')
plt.xlabel('Today\'s Volume')
plt.ylabel('Residuals (Actual - Predicted)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# Optional: Q-Q plot for normality of residuals
sm.qqplot(residuals, line='s')
plt.title(f'Q-Q Plot of Residuals for {TICKER}')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

--- Fetching daily data for MSFT from 2015-01-01 to 2023-12-31 ---


/tmp/ipython-input-2517265068.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_daily = yf.download(TICKER, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed


Error downloading data: Index.droplevel() got an unexpected keyword argument 'axis'
Please ensure the ticker symbol is correct and you have an active internet connection.

--- Linear Regression Results (statsmodels) ---
                            OLS Regression Results                            
Dep. Variable:        Next_Day_Return   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.5795
Date:                Mon, 11 Aug 2025   Prob (F-statistic):              0.447
Time:                        10:21:32   Log-Likelihood:                 5938.2
No. Observations:                2262   AIC:                        -1.187e+04
Df Residuals:                    2260   BIC:                        -1.186e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                

KeyError: 'Volume'